# What we could do

Three families of intervention, each tested on Leonia's actual street network using the public Bridge OD data. For every intervention we show:

- the **target street** and rationale
- how much of the morning bridge-commute traffic it would deflect
- which nearby streets would absorb the deflected traffic

These are *evidence-based screening tests*, not engineering studies. Any chosen intervention would need a fuller signal-timing & safety review.

In [1]:
from pathlib import Path
import os, sys, warnings

import geopandas as gpd
import pandas as pd

REPO = Path.cwd()
while not (REPO / 'leonia_traffic').is_dir() and REPO.parent != REPO:
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
warnings.filterwarnings('ignore')

from leonia_traffic.network.osm_builder import build_or_load_network
from leonia_traffic.assignment import (
    apply_scenarios_to_graph, bridge_od_to_demand, build_assignment_graph,
    run_ue,
)
from leonia_traffic.simulation.scenarios import (
    Closure, LaneReduction, SpeedHumpCalming,
)

CANON = Path('data/processed/streetlight')
DERIVED = Path('data/processed/derived')
nodes, links, _ = build_or_load_network()
G_base = build_assignment_graph(nodes, links)
bod = pd.read_parquet(CANON / 'bridge_od.parquet')
zones = gpd.read_parquet(CANON / 'bridge_od_zones.parquet')
demand = bridge_od_to_demand(bod, zones, G_base, day_type_code=1, day_part_code=2)
base_res = run_ue(G_base, demand, max_iter=30)
base_by_way = base_res.by_osm_way().set_index('osm_way_id')['assigned_volume_vph']

ct = pd.read_parquet(DERIVED / 'cutthrough_index.parquet')
candidates = ct.sort_values('cutthrough_index', ascending=False).head(5)

def test(scenario, label):
    nn, ll = apply_scenarios_to_graph(nodes, links, [scenario])
    G_sc = build_assignment_graph(nn, ll)
    res = run_ue(G_sc, demand, max_iter=25)
    sc_by_way = res.by_osm_way().set_index('osm_way_id')['assigned_volume_vph']
    j = pd.concat([base_by_way.rename('base'),
                    sc_by_way.rename('scen')], axis=1).fillna(0.0)
    j['delta'] = j['scen'] - j['base']
    return {
        'label': label,
        'closed_baseline_vph': float(base_by_way.get(scenario.osm_way_ids[0], 0)),
        'gainers': j[j['delta'] > 5].sort_values('delta', ascending=False).head(5),
        'losers': j[j['delta'] < -5].sort_values('delta').head(5),
    }
print('baseline assignment ready')

baseline assignment ready


## Option A — Close to through-traffic

Apply a turn restriction or modal filter that removes the worst cut-through street from the through-route. Models the upper bound on possible relief.

In [2]:
top1 = candidates.iloc[0]
sc = Closure(name='close-top1', osm_way_ids=[int(top1['osm_way_id'])])
out = test(sc, f'Close {top1["street_name"]} to through-traffic')
from IPython.display import Markdown, display
display(Markdown(f'**Target**: {top1["street_name"]} (cut-through index '
                  f'{top1["cutthrough_index"]:.2f}, Thursday volume '
                  f'{int(top1["thursday_volume"]):,})'))
display(Markdown(f'**Removed Bridge-OD flow**: {out["closed_baseline_vph"]:.0f} vph in '
                  f'morning peak'))
display(Markdown('**Streets that would absorb the diverted traffic:**'))
display(out['gainers'][['delta']].rename(columns={'delta': '+ vph'}))

**Target**: Willow Tree Road (cut-through index 0.59, Thursday volume 1,298)

**Removed Bridge-OD flow**: 0 vph in morning peak

**Streets that would absorb the diverted traffic:**

,+ vph
osm_way_id,


## Option B — Lane reduction

Narrow the street's through capacity (e.g. add bike lane, add parking). Less drastic than closure; models a partial deflection.

In [3]:
sc = LaneReduction(name='lane-red', osm_way_ids=[int(top1['osm_way_id'])],
                    target_lanes=1)
out = test(sc, f'Lane reduction on {top1["street_name"]}')
display(Markdown(f'**Target**: {top1["street_name"]}, reduce to 1 through-lane'))
display(Markdown(f'**Net change on target**: {(out["gainers"]["delta"].sum() + out["losers"]["delta"].sum()):.0f} vph'))
display(Markdown('**Largest changes elsewhere:**'))
display(pd.concat([out['gainers'].head(3), out['losers'].head(3)])[['delta']].rename(
    columns={'delta': 'Δ vph'}))

**Target**: Willow Tree Road, reduce to 1 through-lane

**Net change on target**: 0 vph

**Largest changes elsewhere:**

,Δ vph
osm_way_id,


## Option C — Traffic calming

Speed humps / chicanes that drop the effective free-flow speed by ~50 %. Discourages cut-through without removing the street.

In [4]:
sc = SpeedHumpCalming(name='calm', osm_way_ids=[int(top1['osm_way_id'])],
                       free_flow_speed_factor=0.5)
out = test(sc, f'Speed-hump calming on {top1["street_name"]}')
display(Markdown(f'**Target**: {top1["street_name"]}, halve free-flow speed'))
display(Markdown('**Net diversion:**'))
display(pd.concat([out['gainers'].head(3), out['losers'].head(3)])[['delta']].rename(
    columns={'delta': 'Δ vph'}))

**Target**: Willow Tree Road, halve free-flow speed

**Net diversion:**

,Δ vph
osm_way_id,


## Side-by-side: top 3 candidate streets, closure scenario

In [5]:
rows = []
for _, c in candidates.head(3).iterrows():
    sc = Closure(name=f'c-{c.osm_way_id}', osm_way_ids=[int(c.osm_way_id)])
    out = test(sc, c['street_name'])
    rows.append({
        'Street': c['street_name'],
        'Cut-through index': f'{c["cutthrough_index"]:.2f}',
        'Bridge-flow removed (vph)': f'{out["closed_baseline_vph"]:.0f}',
        'Streets absorbing (>5 vph)': len(out['gainers']),
        'Top absorbing osm_id': (out['gainers'].head(1).index.tolist()[0]
                                  if len(out['gainers']) else '—'),
    })
pd.DataFrame(rows)

,Street,Cut-through index,Bridge-flow removed (vph),Streets absorbing (>5 vph),Top absorbing osm_id
0,Willow Tree Road,0.59,0,0,—
1,Broad Avenue,0.56,0,0,—
2,Schor Avenue,0.55,0,0,—


## Caveats

The numbers above use **Bridge-OD demand only** — i.e. traffic destined for the George Washington Bridge during the morning commute. Streets that carry other cut-through traffic (school drop-off, retail, NJ Turnpike spillover) will show smaller numbers here than they would in a full traffic model. Treat these as *floor estimates* of the diversion impact.

## What's next

[**What we recommend →**](03_what_we_recommend.html)